# Platform requirement audit v1

目标：把当前 SOXL 全平台目标拆成可审计 requirement，并从真实 repo artifact 读取证据。该 notebook 不声明长期目标完成，只记录当前 pass/review/fail 状态。

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "AGENTS.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.core import build_platform_requirement_audit, write_platform_requirement_audit

EXPERIMENT = "platform_requirement_audit_v1"
CARD = ROOT / "research" / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)

audit = build_platform_requirement_audit(ROOT)
write_result = write_platform_requirement_audit(CARD, audit)
summary = {
    "schema_version": "research.platform_requirement_audit_v1",
    "status": audit["summary"]["status"],
    "core_schema_version": audit["schema_version"],
    "summary_schema_version": audit["summary"]["schema_version"],
    "requirement_count": audit["summary"]["requirement_count"],
    "passed_count": audit["summary"]["passed_count"],
    "review_count": audit["summary"]["review_count"],
    "failed_count": audit["summary"]["failed_count"],
    "review_requirement_ids": audit["summary"]["review_requirement_ids"],
    "failed_requirement_ids": audit["summary"]["failed_requirement_ids"],
    "completion_claim": audit["summary"]["completion_claim"],
    "artifacts": {
        "audit": "platform_requirement_audit.json",
        "audit_csv": "platform_requirement_audit.csv",
        "audit_review": "platform_requirement_audit_review.md",
    },
    "write_result": write_result,
}
(CARD / "platform_requirement_audit_summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
summary


{'schema_version': 'research.platform_requirement_audit_v1',
 'status': 'review',
 'core_schema_version': 'core.platform_requirement_audit.v1',
 'summary_schema_version': 'core.platform_requirement_audit.summary.v1',
 'requirement_count': 13,
 'passed_count': 11,
 'review_count': 2,
 'failed_count': 0,
 'review_requirement_ids': ['production_readiness_review_gate',
  'external_knowledge_promotion_backlog'],
 'failed_requirement_ids': [],
 'completion_claim': 'not_claimed',
 'artifacts': {'audit': 'platform_requirement_audit.json',
  'audit_csv': 'platform_requirement_audit.csv',
  'audit_review': 'platform_requirement_audit_review.md'},
 'write_result': {'schema_version': 'core.platform_requirement_audit_write_result.v1',
  'audit_path': '/Users/rziiiii/dev/workspace/research/platform_requirement_audit_v1/platform_requirement_audit.json',
  'csv_path': '/Users/rziiiii/dev/workspace/research/platform_requirement_audit_v1/platform_requirement_audit.csv',
  'review_path': '/Users/rziiiii/

In [2]:
assert audit["schema_version"] == "core.platform_requirement_audit.v1"
assert summary["schema_version"] == "research.platform_requirement_audit_v1"
assert summary["status"] == "review"
assert summary["failed_count"] == 0
assert summary["review_count"] == 2
assert summary["completion_claim"] == "not_claimed"
assert summary["review_requirement_ids"] == [
    "production_readiness_review_gate",
    "external_knowledge_promotion_backlog",
]
assert (CARD / "platform_requirement_audit.json").exists()
assert (CARD / "platform_requirement_audit.csv").exists()
assert (CARD / "platform_requirement_audit_review.md").exists()
summary


{'schema_version': 'research.platform_requirement_audit_v1',
 'status': 'review',
 'core_schema_version': 'core.platform_requirement_audit.v1',
 'summary_schema_version': 'core.platform_requirement_audit.summary.v1',
 'requirement_count': 13,
 'passed_count': 11,
 'review_count': 2,
 'failed_count': 0,
 'review_requirement_ids': ['production_readiness_review_gate',
  'external_knowledge_promotion_backlog'],
 'failed_requirement_ids': [],
 'completion_claim': 'not_claimed',
 'artifacts': {'audit': 'platform_requirement_audit.json',
  'audit_csv': 'platform_requirement_audit.csv',
  'audit_review': 'platform_requirement_audit_review.md'},
 'write_result': {'schema_version': 'core.platform_requirement_audit_write_result.v1',
  'audit_path': '/Users/rziiiii/dev/workspace/research/platform_requirement_audit_v1/platform_requirement_audit.json',
  'csv_path': '/Users/rziiiii/dev/workspace/research/platform_requirement_audit_v1/platform_requirement_audit.csv',
  'review_path': '/Users/rziiiii/

## 中文结论

- 做了什么：新增平台需求覆盖 audit，把长期 SOXL 全平台目标拆成 stable src surfaces、objective/reward 退役、notebook contract、QuantBench registry、训练回测 ledger、analysis、dashboard、data、schedule、paper API、knowledge ingest 和 skill boundary 等可审计 requirement。
- 核心指标：当前 audit status 为 `review`，共 13 条 requirement，其中 11 条 pass、2 条 review、0 条 fail；`completion_claim=not_claimed`，因此不能把长期目标标记为完成。
- regime 差异：本 notebook 不重新计算市场 regime；它检查 production `analysis_manifest` 是否已包含 regime/window/model difference/failure/objective loss 等证据面。
- 是否晋升：否。本卡是 research audit，不是 production 策略或模型晋升；它只提供当前架构完成度证据。
- 风险：audit 只能证明当前文件和 manifest 证据是否一致，不能替代人工 review；未来数据、模型或 production artifact 变更后必须重新执行。
- 下一步：优先处理 review 项：`production_readiness_review_gate` 和 `external_knowledge_promotion_backlog`，分别对应 objective loss review gate 与仍 blocked 的外部知识候选。